# Experiment

In [1]:
0

0

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import gc
import os
import pathlib
import subprocess
import sys
import pickle
import shutil
import json
import glob
import re
import tempfile
from pathlib import Path
from typing import Iterable

docs_dir = str(pathlib.Path(os.getcwd()).resolve().parents[0])
sys.path.append(docs_dir)
print(docs_dir)

from src.install_cellranger import install_cellranger
from src.download_references import download_references
from src.build_cellranger_mkref import build_cellranger_mkref
from src.fastq_datasets import fastq_datasets
from src.download_fastq import download_fastq
from src.run_cellranger_master import existing_outs_for_release
from src.anndata_generator import anndata_generator

/ictstr01/home/icb/kemal.inecik/work/codes/idtrack/docs/_notebooks


In [4]:
# %matplotlib inline
# %config InlineBackend.figure_format='retina'

# import matplotlib.pyplot as plt
# import seaborn as sns

# # Important to have consistent figures across platforms:
# _rcparams_path = os.path.join(docs_dir, "figure_rcparams", "rcparams.pickle")
# with open(_rcparams_path, "rb") as file:
#     _rcparams = pickle.load(file)
# plt.rcParams.update(_rcparams)

# Preparation

## Install cellranger

In [5]:
cellranger_params = dict(
    version="9.0.1",
    expected_md5="2efec98bff01f7a59edaf43724fae13f",
    url="https://cf.10xgenomics.com/releases/cell-exp/cellranger-9.0.1.tar.gz?Expires=1756836323&Key-Pair-Id=APKAI7S6A5RYOXBWRPDA&Signature=kjLqu7Mer5A0F4hV1PlJQXFGoY0pJZC8OsPilkGkIBOdxG~crQI5eLMSvV~cZzx0u94PrhZvlXofddsjzhvFXe3mPd2bauIyP5RxITf7QgpHWeZ6hNFhLI9CD5jJXgGs0bzhNhSEquxlLjHH~W3v7zpwk5mzV7jgD7tCwd00C4q-yRtwNefer~ZV7p01FwOIlo6dckef7dMgDIb~6FBuGlkDbyChbJOWITXSGyxz7IwRkFIt9C4X2bsPQ29n3I0gX3zVIAbavgKVC0ZOk0lUx6SucGUQZbhQeTvLysWDhRldj7TrYn9ZoJpo8fz50JMSTn9o3QNatz0rBs-DQasyNg__",
    apps_dir=pathlib.Path("/home/icb/kemal.inecik/tools/apps").expanduser(),
    tmp_dir=pathlib.Path("/home/icb/kemal.inecik/tools/tmp").expanduser(),
)

In [5]:
cellranger_bin = install_cellranger(**cellranger_params)

[cellranger] Downloading with curl to /ictstr01/home/icb/kemal.inecik/tools/tmp/cellranger-9.0.1.tar.gz ...
[cellranger] MD5 (python) = 2efec98bff01f7a59edaf43724fae13f
[cellranger] MD5 (md5sum) = 2efec98bff01f7a59edaf43724fae13f
[cellranger] Extracting with tar to /ictstr01/home/icb/kemal.inecik/tools/apps ...
[cellranger] Done. Installed at: /ictstr01/home/icb/kemal.inecik/tools/apps/cellranger-9.0.1
[cellranger] Tarball kept at: /ictstr01/home/icb/kemal.inecik/tools/tmp/cellranger-9.0.1.tar.gz


In [6]:
cellranger_bin = install_cellranger(**cellranger_params)
os.environ["PATH"] = f"{cellranger_bin.parent}:{os.environ['PATH']}"
print("bin:", cellranger_bin)
print("version:", subprocess.run([str(cellranger_bin), "--version"], text=True, capture_output=True).stdout)

[cellranger] Found existing install: /ictstr01/home/icb/kemal.inecik/tools/apps/cellranger-9.0.1/cellranger
bin: /ictstr01/home/icb/kemal.inecik/tools/apps/cellranger-9.0.1/cellranger
version: cellranger cellranger-9.0.1



## Download annotation sources

In [7]:
download_references_params = dict(
    workdir = pathlib.Path("/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/references_gold_standard"),
    releases = range(80, 115),  # 80..114 inclusive
    fasta_name = "Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz",
    gunzip=True,        
    keep_gz=True,       
)

In [7]:
paths_references = download_references(silent=False, **download_references_params)

[release 80] Downloaded: Homo_sapiens.GRCh38.80.gtf.gz
[release 80] Gunzipped: Homo_sapiens.GRCh38.80.gtf
[release 80] Downloaded: Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
[release 80] Renamed: Homo_sapiens.GRCh38.80.dna.primary_assembly.fa.gz
[release 80] Gunzipped: Homo_sapiens.GRCh38.80.dna.primary_assembly.fa
[release 81] Downloaded: Homo_sapiens.GRCh38.81.gtf.gz
[release 81] Gunzipped: Homo_sapiens.GRCh38.81.gtf
[release 81] Downloaded: Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
[release 81] Renamed: Homo_sapiens.GRCh38.81.dna.primary_assembly.fa.gz
[release 81] Gunzipped: Homo_sapiens.GRCh38.81.dna.primary_assembly.fa
[release 82] Downloaded: Homo_sapiens.GRCh38.82.gtf.gz
[release 82] Gunzipped: Homo_sapiens.GRCh38.82.gtf
[release 82] Downloaded: Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
[release 82] Renamed: Homo_sapiens.GRCh38.82.dna.primary_assembly.fa.gz
[release 82] Gunzipped: Homo_sapiens.GRCh38.82.dna.primary_assembly.fa
[release 83] Downloaded: Homo_sapiens

In [8]:
paths_references = download_references(silent=True, **download_references_params)

## Create reference for cellranger

In [9]:
build_cellranger_mkref_params = dict(
    paths_by_release = paths_references,
    cellranger_bin = cellranger_bin,
    assembly_label = "GRCh38",
    use_threads = 28
)

In [10]:
paths_cellranger_mkref = build_cellranger_mkref(silent=False, **build_cellranger_mkref_params)

[release 80] Running: /ictstr01/home/icb/kemal.inecik/tools/apps/cellranger-9.0.1/bin/cellranger mkgtf /lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/references_gold_standard/GRCh38_80/Homo_sapiens.GRCh38.80.gtf /lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/references_gold_standard/GRCh38_80/Homo_sapiens.GRCh38.80.filtered.gtf --attribute=gene_biotype:protein_coding --attribute=gene_type:protein_coding
[release 80] mkgtf OK: Homo_sapiens.GRCh38.80.filtered.gtf
[release 80] Running: /ictstr01/home/icb/kemal.inecik/tools/apps/cellranger-9.0.1/bin/cellranger mkref --genome=reference_GRCh38_80 --fasta=/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/references_gold_standard/GRCh38_80/Homo_sapiens.GRCh38.80.dna.primary_assembly.fa --genes=/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/references_gold_standard/GRCh38_80/Homo_sapiens.GRCh38.80.filtered.gtf --nthreads=28 (cwd=/lustre/groups/ml01/workspace/kemal.inecik/idtrack_exper

In [10]:
paths_cellranger_mkref = build_cellranger_mkref(silent=True, **build_cellranger_mkref_params)

## Download sequencing data

In [11]:
download_fastq_params = dict(
    datasets = fastq_datasets,
    working_dir = pathlib.Path("/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/fastq_raw"),
)

In [ ]:
paths_download_fastq = download_fastq(silent=False, **download_fastq_params)

Downloading: 'pbmc_1k_v3'
Downloading: 'pbmc_20k_donors1_4_multiplex_gemx_3p'
Downloading: 'pbmc_10k_5p_v3_ultima'
Downloading: 'pbmc_10k_3p_v31_si'


In [12]:
paths_download_fastq = download_fastq(silent=True, **download_fastq_params)

## Run alignments

In [13]:
override = False
datasets = ["pbmc_1k_v3"]  # you can add more later
assembly_labels = ["GRCh38"]

prior_workdir = Path("/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments").resolve()
working_dir = Path("/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments").resolve()
working_dir.mkdir(parents=True, exist_ok=True)
run_cellranger_script = Path.cwd().resolve().parent / "src" / "run_cellranger_master.py"

count = 0
errors = []

for dataset_key in paths_download_fastq:

    if dataset_key not in datasets:
        continue
    
    # (A) discover releases for the first (and only) assembly in your note, but we allow multiple
    for assembly_label in assembly_labels:
        
        for release, release_content in paths_cellranger_mkref.items():

            # if count > 0:
            #     break

            if release_content["status"] != 'done':
                print(f"[WARN] No releases discovered for assembly {assembly_label!r}, release {release!r}.")
                continue
            
            # Per your request: create a folder for each dataset / release / assembly under working_dir
            job_dir = (working_dir / dataset_key / f"{assembly_label}_{release}").resolve()
            job_dir.mkdir(parents=True, exist_ok=True)
            # Slurm log lives in the "respective" folder
            slurm_log = job_dir / f"slurm_job.log"
            # Skip if existing output (unless override)
            existing = existing_outs_for_release(working_dir, dataset_key, assembly_label, release)
            if existing and not override:
                print(f"[SKIP] Found existing metrics for {dataset_key} release {release}: {existing}")
                continue

            # Build Slurm script (kept inside the notebook cell, per your example)
            slurm_script = f"""#!/bin/bash
#SBATCH -J cr_{dataset_key}_r{release}
#SBATCH -p cpu_p
#SBATCH --qos cpu_normal
#SBATCH -c 4
#SBATCH --mem=32G
#SBATCH --nice=0
#SBATCH -t 11:50:00
#SBATCH -o {slurm_log}
#SBATCH -e {slurm_log}

echo "Start"

set -eo pipefail

echo "Environment setup"
source ~/.bashrc && conda activate idtrack_dev_env

echo "Python started"
python -u "{run_cellranger_script}" \\
  --dataset-key "{dataset_key}" \\
  --release {release} \\
  --workdir "{prior_workdir}" \\
  --results-dir "{working_dir}" \\
  --assembly-label "{assembly_label}" \\
  --cellranger-bin "{cellranger_bin}" \\
  --no-download

echo "End."
"""
            try:
                script_name = job_dir / f"slurm_job.sh"
                with open(script_name, "w") as f:
                    f.write(slurm_script)
    
                print(f"Submitting: {dataset_key} / {assembly_label} / r{release} / log {slurm_log}")
                subprocess.run(["sbatch", str(script_name)], check=False)
                count += 1
        
            finally:
                # os.remove(script_name)
                pass
                
print(f" - Number of jobs submitted: {count}")

[WARN] No releases discovered for assembly 'GRCh38', release 80.
[WARN] No releases discovered for assembly 'GRCh38', release 81.
Submitting: pbmc_1k_v3 / GRCh38 / r82 / log /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_82/slurm_job.log
Submitted batch job 30744789
Submitting: pbmc_1k_v3 / GRCh38 / r83 / log /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_83/slurm_job.log
Submitted batch job 30744790
Submitting: pbmc_1k_v3 / GRCh38 / r84 / log /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_84/slurm_job.log
Submitted batch job 30744791
Submitting: pbmc_1k_v3 / GRCh38 / r85 / log /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_85/slurm_job.log
Submitted batch job 30744792
Submitting: pbmc_1k_v3 / GRCh38 / r86 / log /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRC

## Prepare anndata

In [13]:
anndata_generator_params = dict(
    alignments_root=Path("/lustre/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments"),
    out_root="/home/icb/kemal.inecik/lustre_workspace/idtrack_experiments/anndatas",
    keep_obs_if_it_is_filtered_by_cellranger=True
)

In [14]:
anndata_paths = anndata_generator(silent=False, **anndata_generator_params)

[INFO] Building AnnData from /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_100/pbmc_1k_v3_count_r100/outs (run=pbmc_1k_v3_count_r100) [filtered_only=True]
[OK] Wrote pbmc_1k_v3_GRCh38_100.h5ad  shape=1221x19970
[INFO] Building AnnData from /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_101/pbmc_1k_v3_count_r101/outs (run=pbmc_1k_v3_count_r101) [filtered_only=True]
[OK] Wrote pbmc_1k_v3_GRCh38_101.h5ad  shape=1221x19966
[INFO] Building AnnData from /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_102/pbmc_1k_v3_count_r102/outs (run=pbmc_1k_v3_count_r102) [filtered_only=True]
[OK] Wrote pbmc_1k_v3_GRCh38_102.h5ad  shape=1222x19973
[INFO] Building AnnData from /ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_103/pbmc_1k_v3_count_r103/outs (run=pbmc_1k_v3_count_r103) [filtered_only=True]
[OK] Wrote pbmc_1k_v

In [18]:
anndata_paths = anndata_generator(silent=True, **anndata_generator_params)

In [21]:
import anndata as ad
adata114=ad.read_h5ad(anndata_paths['pbmc_1k_v3']['GRCh38']['100'])

In [22]:
adata114

AnnData object with n_obs × n_vars = 1221 × 19970
    obs: 'is_filtered_by_cellranger'
    var: 'gene_id', 'gene_symbol', 'feature_type', 'genome'
    uns: 'cellranger', 'provenance'

In [23]:
adata114.obs

,is_filtered_by_cellranger
barcode,
AAACCCAAGGAGAGTA-1,True
AAACGCTTCAGCCCAG-1,True
AAAGAACAGACGACTG-1,True
AAAGAACCAATGGCAG-1,True
AAAGAACGTCTGCAAT-1,True
...,...
TTTCCTCTCTCTTGCG-1,True
TTTGATCTCTTTGGAG-1,True
TTTGGTTAGTAACCTC-1,True


In [24]:
adata114.var

,gene_id,gene_symbol,feature_type,genome
feature_index,,,,
0,ENSG00000186092,OR4F5,Gene Expression,reference_GRCh38_100
1,ENSG00000284733,OR4F29,Gene Expression,reference_GRCh38_100
2,ENSG00000284662,OR4F16,Gene Expression,reference_GRCh38_100
3,ENSG00000187634,SAMD11,Gene Expression,reference_GRCh38_100
4,ENSG00000188976,NOC2L,Gene Expression,reference_GRCh38_100
...,...,...,...,...
19965,ENSG00000277856,AC233755.2,Gene Expression,reference_GRCh38_100
19966,ENSG00000275063,AC233755.1,Gene Expression,reference_GRCh38_100
19967,ENSG00000271254,AC240274.1,Gene Expression,reference_GRCh38_100


In [25]:
adata114.uns

{'cellranger': {'assembly_label': 'GRCh38',
  'cloupe_file': '/ictstr01/groups/ml01/workspace/kemal.inecik/idtrack_experiments/alignments/pbmc_1k_v3/GRCh38_100/pbmc_1k_v3_count_r100/outs/cloupe.cloupe',
  'dataset': 'pbmc_1k_v3',
  'ensembl_release': 100,
  'metrics_summary': {'Estimated Number of Cells': '1,221',
   'Fraction Reads in Cells': '95.6%',
   'Mean Reads per Cell': '54,547',
   'Median Genes per Cell': '2,985',
   'Median UMI Counts per Cell': '9,188',
   'Number of Reads': '66,601,887',
   'Q30 Bases in Barcode': '94.1%',
   'Q30 Bases in RNA Read': '90.2%',
   'Q30 Bases in UMI': '92.7%',
   'Reads Mapped Antisense to Gene': '7.9%',
   'Reads Mapped Confidently to Exonic Regions': '53.9%',
   'Reads Mapped Confidently to Genome': '93.3%',
   'Reads Mapped Confidently to Intergenic Regions': '8.9%',
   'Reads Mapped Confidently to Intronic Regions': '30.5%',
   'Reads Mapped Confidently to Transcriptome': '74.5%',
   'Reads Mapped to Genome': '96.1%',
   'Sequencing Satur

# Analysis

## Running IDTrack

In [ ]:
pass

## Analysis of IDTrack performance

In [ ]:
pass

### Finalized figures

In [ ]:
pass

## Running competing tools

In [ ]:
pass

## Analysis of IDTrack performance

In [ ]:
pass

### Finalized figures

In [ ]:
pass